In [1]:
import pandas as pd 
artifact_path = "../artifacts/ml_table.csv"
df=pd.read_csv(artifact_path)
print("Shape:",df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (99441, 22)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_items_price', 'total_freight_value', 'unique_products', 'unique_sellers', 'payment_count', 'total_payment_value', 'payment_types', 'max_installments', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'review_score']


In [2]:
data_columns=[
        "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
print(df[data_columns].dtypes)

order_purchase_timestamp         object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [3]:
df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"],
    errors="coerce"
)

df["order_estimated_delivery_date"] = pd.to_datetime(
    df["order_estimated_delivery_date"],
    errors="coerce"
)

print(df[
    ["order_delivered_customer_date", "order_estimated_delivery_date"]
].dtypes)

order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [4]:
print("Missing delivered dates:",
      df["order_delivered_customer_date"].isna().sum())

print("Missing estimated dates:",
      df["order_estimated_delivery_date"].isna().sum())

Missing delivered dates: 2965
Missing estimated dates: 0


In [5]:
missing_delivery = df[
    df["order_delivered_customer_date"].isna()
]

print("Orders with missing delivery date:",
      len(missing_delivery))

print("\nOrder status distribution:")
print(missing_delivery["order_status"].value_counts(dropna=False))

Orders with missing delivery date: 2965

Order status distribution:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [6]:
delivered_orders = df[
    df["order_delivered_customer_date"].notna()
]

print("Orders with delivery date:", len(delivered_orders))
print("\nStatus distribution:")
print(delivered_orders["order_status"].value_counts(dropna=False))

Orders with delivery date: 96476

Status distribution:
order_status
delivered    96470
canceled         6
Name: count, dtype: int64


In [7]:
# Create label only for orders with a known delivery date

df["late"] = pd.NA

has_delivery_date = df["order_delivered_customer_date"].notna()

df.loc[has_delivery_date, "late"] = (
    df.loc[has_delivery_date, "order_delivered_customer_date"]
    > df.loc[has_delivery_date, "order_estimated_delivery_date"]
).astype(int)

print(df["late"].value_counts(dropna=False))

late
0       88649
1        7827
<NA>     2965
Name: count, dtype: int64


In [8]:
# Check a few orders manually

check_columns = [
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "late"
]

print(
    df[df["late"].notna()][check_columns]
    .sample(10, random_state=42)
    .sort_values("order_delivered_customer_date")
    .to_string(index=False)
)

                        order_id order_delivered_customer_date order_estimated_delivery_date late
4ef8f514f95bb0a41f58965ea04e7027           2017-05-26 15:59:47                    2017-06-07    0
587904dc1c873ebfb6078160b4819d8e           2017-12-06 19:43:34                    2017-12-15    0
80b430d0029bb33110ac31d60e87e0b8           2017-12-07 18:43:46                    2017-12-27    0
462240cb1ec4e5db517e73fff57ebfc0           2017-12-14 18:56:14                    2017-12-20    0
5bc2a7b8f0817443a86b69461e742cc9           2018-01-12 21:59:18                    2018-02-01    0
c58cff333993bb6b7161d7ec1350eef3           2018-04-06 02:32:49                    2018-04-18    0
c09f32e7ba9b4a134455b36eeff8fff3           2018-04-13 17:32:07                    2018-04-24    0
580603672a21252f21fa8a8b4ca85986           2018-04-26 17:44:27                    2018-05-08    0
29c3b79aace1b72a82b1232bf494e16f           2018-04-28 15:51:50                    2018-01-24    1
87673b5ccb20de0a91c2

In [9]:
# Verify that the label matches the date comparison

expected_late = (
    df["order_delivered_customer_date"]
    > df["order_estimated_delivery_date"]
)

check = df["late"].notna()

is_correct = (
    df.loc[check, "late"].astype(int)
    == expected_late.loc[check].astype(int)
)

print("Correct labels:", is_correct.sum())
print("Incorrect labels:", (~is_correct).sum())
print("Total checked:", len(is_correct))

Correct labels: 96476
Incorrect labels: 0
Total checked: 96476


In [10]:
label_counts = df["late"].value_counts()

label_percentages = (
    df["late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Class distribution:")
print(label_counts)

print("\nClass percentages:")
print(label_percentages)

Class distribution:
late
0    88649
1     7827
Name: count, dtype: int64

Class percentages:
late
0    91.89
1     8.11
Name: proportion, dtype: float64


In [11]:
on_time_count = (df["late"] == 0).sum()
late_count = (df["late"] == 1).sum()

imbalance_ratio = on_time_count / late_count

print(f"On-time orders: {on_time_count}")
print(f"Late orders: {late_count}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

On-time orders: 88649
Late orders: 7827
Imbalance ratio: 11.33:1


In [12]:
labeled_df = df[df["late"].notna()].copy()

labeled_df["late"] = labeled_df["late"].astype(int)

print("Labeled dataset shape:", labeled_df.shape)
print("Missing labels:", labeled_df["late"].isna().sum())

Labeled dataset shape: (96476, 23)
Missing labels: 0


In [13]:
output_path = "../artifacts/labeled_table.csv"

labeled_df.to_csv(output_path, index=False)

print(f"Artifact saved successfully: {output_path}")

Artifact saved successfully: ../artifacts/labeled_table.csv


In [14]:
import os
import pandas as pd

artifact_path = "../artifacts/labeled_table.csv"

print("File exists:", os.path.exists(artifact_path))

check_df = pd.read_csv(artifact_path)

print("Shape:", check_df.shape)
print("Unique orders:", check_df["order_id"].nunique())
print("Missing labels:", check_df["late"].isna().sum())
print("\nLabel distribution:")
print(check_df["late"].value_counts())

File exists: True
Shape: (96476, 23)
Unique orders: 96476
Missing labels: 0

Label distribution:
late
0    88649
1     7827
Name: count, dtype: int64
